# Unit 8 solutions: the two-tower model in PyTorch

These reference solutions re-express Unit 5's matrix factorization as a learned neural network. Unit 5 learned a reader taste vector and a book taste vector by full-batch numpy gradient descent and scored a match with their **dot product**. The two-tower keeps *exactly* that latent dot-product model -- a reader embedding tower and a book embedding tower whose dot product scores a match -- but trains the vectors as `torch.nn.Embedding` layers by autograd with the **Adam** optimizer over mini-batches, under a **pairwise ranking** objective (BPR) instead of Unit 5's pointwise logistic loss. This is the first PyTorch unit.

We build the two-tower score and a single BPR step **by hand** on tiny tensors, then train the real `TwoTowerRetrievalPath`, watch its BPR loss fall, register it, and read its validation scoreboard honestly. The two Challenge exercises show *why* the result holds: a weight-decay sweep (the regularization lesson) and the equivalence between the two-tower and Unit 5's MF.

Read the numbers **honestly, and the honest reading is a real win this time**. Measured on this data (seed 0, k=10, 500 validation readers), the regularized two-tower scores hit@10 ~0.340 -- the **best path in the book so far**, above MF (~0.276), item-item CF (~0.252), lexical (~0.158) and the random floor (~0.012). It is **not** that "neural is always better": the two-tower is the *same* reader-times-book dot product as MF. What changed is the *objective and optimizer* -- a pairwise **BPR ranking** loss with Adam mini-batches and **weight decay**, versus MF's pointwise logistic full-batch gradient descent -- and that ranking objective wins on top-k, but **only when regularized**. With `weight_decay=0` the BPR loss collapses toward 0 (the model memorizes the train pairs) and val hit@10 drops below item-item CF (Exercise 6 measures this directly).

Everything is measured on the **validation** split (`val`); each reader's **seen** set is their train-positive history. The `test` holdout stays sealed -- no cell here touches it.

Two speed notes, because this is the first unit that trains a neural network. A full `TwoTowerRetrievalPath` fit (the pinned 60 epochs) takes ~15 s; we fit it a few times and **reuse** the fitted objects, and every training cell stays well under the 120 s/cell budget. The weight-decay sweep in Exercise 6 does four fits, so it trains for only 20 epochs each (stated in that cell) to stay fast. Every torch cell imports torch inside the cell and seeds it (`torch.manual_seed(0)` + deterministic algorithms + a single thread), the reproducibility rule for this unit.

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    PathRegistry,
    RandomRetrievalPath,
    TwoTowerRetrievalPath,
    rank,
    run_validation_scoreboard,
)

SEED = 0
K = 10        # every scoreboard reports at the top-10 cutoff

# --- the real interaction log (the same substrate as Units 1-7) ----------------------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)                      # 2000 books
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")  # {item_id: "space joined token bag"}

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])           # 60 readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

# Each reader's train-positive history is their `seen` (exclusion) set; their val positives are the
# relevance target. We grade on `val`; no cell here reads the sealed `test` split.
seen_by = defaultdict(set)
val_pos_by = defaultdict(set)
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        val_pos_by[row["reader_id"]].add(row["item_id"])

## Exercise 1 - The two-tower score IS a dot product

A two-tower model gives every reader one vector (from the **reader tower**) and every book one vector (from the **book tower**), both living in the same `embedding_dim`-dimensional space. The predicted match is the simplest possible thing: their **dot product**, `score(reader, book) = sum(reader_emb * book_emb)`. A book whose vector points the same way as the reader's scores high; one pointing the opposite way scores low. That is the entire scoring rule -- the same dot product as Unit 5's MF and Unit 7's cosine, the whole embedding-retrieval family.

Work it by hand on tiny tensors. Given one reader vector and a small matrix of three book vectors, score each book by the dot product (an elementwise product, then a sum over the embedding dimension), rank the books best-first, and confirm your tensor computation matches the dot product you would write by hand with numpy. The reader below points most like book A and opposite to book B, so the ranking should put A first and B last. In two or three sentences, say why a dot product is a natural "match score" for two vectors in a shared space (hint: it rewards agreement coordinate by coordinate).

In [ ]:
import torch

# This cell uses torch directly, so seed it and pin determinism (the reproducibility rule for every
# torch cell): a fixed seed + deterministic algorithms + a single thread.
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)

# One reader vector and three book vectors in a tiny 4-dimensional embedding space.
reader_vec = torch.from_numpy(np.array([0.5, -0.2, 0.8, 0.1], dtype=np.float32))
book_mat = torch.from_numpy(np.array([
    [0.4, -0.1, 0.9, 0.0],    # book 0 (A): points the same way as the reader
    [-0.6, 0.3, -0.2, 0.5],   # book 1 (B): points the opposite way
    [0.1, 0.1, 0.1, 0.1],     # book 2 (C): neutral
], dtype=np.float32))

# Score each book by the dot product reader_vec . book_vec: an elementwise product, then a sum
# over the embedding dimension (dim=1 collapses the 4 coordinates, leaving one score per book row).
scores = (book_mat * reader_vec).sum(dim=1)      # one score per book row
scores_np = scores.numpy()                        # a length-3 numpy array

# Rank the book rows best-first and check the aligned book (row 0) wins, the opposite (row 1) loses.
ranking = list(np.argsort(-scores_np))            # expect [0, 2, 1]
top_is_aligned = bool(ranking[0] == 0)
bottom_is_opposite = bool(ranking[-1] == 1)

# The same number, computed as a plain dot product by hand for book A -- the score IS a dot product.
hand_dot_a = float(np.sum(
    np.array([0.5, -0.2, 0.8, 0.1], dtype=np.float32)
    * np.array([0.4, -0.1, 0.9, 0.0], dtype=np.float32)))
matches_tensor = bool(np.isclose(hand_dot_a, float(scores_np[0]), atol=1e-6))

dot_product_explanation = (
    "A dot product multiplies the two vectors coordinate by coordinate and sums the results, so it "
    "rewards AGREEMENT one axis at a time: where the reader and the book both have a large value "
    "(or both negative), that coordinate adds a big positive term, and where they point opposite "
    "ways it subtracts. Two vectors pointing the same direction in the shared embedding space "
    "therefore score high and opposite vectors score low -- a natural 'match' score for a reader "
    "and a book living in one space."
)

print("scores:", [round(float(s), 3) for s in scores_np])
print("ranking (best first):", ranking)
print(f"hand dot for book A = {hand_dot_a:.3f}  == tensor score {float(scores_np[0]):.3f}? {matches_tensor}")

assert ranking == [0, 2, 1], "aligned book A first, neutral C middle, opposite book B last"
assert top_is_aligned, "the book pointing the reader's way must rank first"
assert bottom_is_opposite, "the book pointing opposite must rank last"
assert matches_tensor, "the tensor score IS the by-hand dot product"

## Exercise 2 - One BPR step pushes a positive above a negative

A read or a like is a **positive**; there are no negative ratings. So the two-tower trains on *pairs*: for a reader, a book they read (a **positive**) should score **above** a book they did not (a sampled **negative**). That is **BPR** (Bayesian Personalized Ranking), the pairwise loss `-log sigmoid(s_pos - s_neg)`. When the positive already beats the negative by a wide margin the loss is near 0; when the negative wins, the loss is large. Minimizing it nudges the embeddings so `s_pos - s_neg` grows -- a *ranking* objective, unlike Unit 5's pointwise logistic loss that pushed each score toward a 0/1 label.

Build the smallest possible example: one reader, two books (book 0 is the positive, book 1 the sampled negative), each a learned `torch.nn.Embedding`. Compute the BPR loss, take a **single Adam step** on it, and confirm two things fall into place: the loss goes **down**, and the margin `s_pos - s_neg` goes **up**. In two or three sentences, contrast this pairwise ranking loss with Unit 5's pointwise logistic loss -- what does BPR care about that a per-item label does not?

In [ ]:
import torch

# A torch cell: seed it and pin determinism so the random embedding init is reproducible.
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)

# Tiny towers: 1 reader, 2 books, a 4-dim embedding. Small init keeps the first scores near 0.
reader_tower = torch.nn.Embedding(1, 4)
item_tower = torch.nn.Embedding(2, 4)
torch.nn.init.normal_(reader_tower.weight, mean=0.0, std=0.1)
torch.nn.init.normal_(item_tower.weight, mean=0.0, std=0.1)
optimizer = torch.optim.Adam(
    list(reader_tower.parameters()) + list(item_tower.parameters()), lr=0.1
)

r_idx = torch.from_numpy(np.array([0]))      # the one reader
pos_idx = torch.from_numpy(np.array([0]))    # the positive book
neg_idx = torch.from_numpy(np.array([1]))    # the sampled negative book


def scores_pos_neg():
    """Return (s_pos, s_neg) as 1-element tensors: each a reader . book dot product."""
    reader_vec = reader_tower(r_idx)
    s_pos = (reader_vec * item_tower(pos_idx)).sum(dim=1)
    s_neg = (reader_vec * item_tower(neg_idx)).sum(dim=1)
    return s_pos, s_neg


def bpr_loss():
    s_pos, s_neg = scores_pos_neg()
    # BPR: -log sigmoid(s_pos - s_neg). Near 0 when the positive already beats the negative widely.
    return -torch.nn.functional.logsigmoid(s_pos - s_neg).mean()


def margin():
    s_pos, s_neg = scores_pos_neg()
    return float((s_pos - s_neg).item())


# Measure BEFORE the step (loss.item() reads the scalar off the graph without the grad-tensor warning).
loss_before = float(bpr_loss().item())
margin_before = margin()

# One Adam step on the BPR loss: zero the old gradients, backprop, step.
optimizer.zero_grad()
step_loss = bpr_loss()
step_loss.backward()
optimizer.step()

# Measure AFTER the step.
loss_after = float(bpr_loss().item())
margin_after = margin()

loss_fell = bool(loss_after < loss_before)
margin_grew = bool(margin_after > margin_before)

bpr_explanation = (
    "BPR is a PAIRWISE ranking loss: it cares only that the positive book outranks the sampled "
    "negative -- that the margin s_pos - s_neg is large and positive -- and is indifferent to each "
    "book's absolute score. Unit 5's POINTWISE logistic loss instead pushed each book's score "
    "toward its own 0/1 label, judging books one at a time. The single Adam step here grows the "
    "margin and so lowers the loss, nudging the embeddings so the read book sits above the unread one."
)

print(f"loss  {loss_before:.4f} -> {loss_after:.4f}  (fell? {loss_fell})")
print(f"margin {margin_before:.4f} -> {margin_after:.4f}  (grew? {margin_grew})")

assert loss_fell, "one BPR step must lower the loss"
assert margin_grew, "one BPR step must push the positive further above the negative"
assert loss_before > 0.0, "a nonzero starting loss (the positive did not already dominate)"

## Exercise 3 - Train the two-tower and watch the BPR loss fall

Now the real thing. `TwoTowerRetrievalPath` trains a reader `nn.Embedding` tower and a book `nn.Embedding` tower on the whole interaction log by BPR + Adam, exactly like the two-book toy in Exercise 2 but with every reader, the full catalog, and `n_negatives` sampled negatives per positive each epoch. Its constructor defaults are **pinned** to the configuration measured on this data: `embedding_dim=32, n_epochs=60, learning_rate=0.01, n_negatives=10, batch_size=256, weight_decay=1e-4, seed=0`. The `fit(rows, catalog=catalog_ids)` call keeps the exact protocol signature (torch is imported lazily inside `fit`, nowhere else), and it records the mean BPR loss of each epoch in `loss_history`.

Fit it once (~15 s) and keep the fitted object -- later exercises reuse it. Read `loss_history` and confirm the loss **fell** from the first epoch to the last (the towers learned to rank). Then confirm the fitted path serves a warm reader: a reader with a learned embedding gets `K` recommendations. In two or three sentences, explain what a falling BPR loss means in ranking terms (positives are being pushed above their sampled negatives) -- and note that a falling training loss alone does **not** prove the model generalizes (Exercise 6 shows a loss that falls to ~0 while validation accuracy collapses).

In [ ]:
# One full fit of the two-tower (~15 s). REUSED by Exercises 4, 5 and 7 -- do not refit there.
# torch is imported lazily INSIDE fit (nowhere in this module's import path), so this is the first
# cell that actually trains a network.
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)

losses = two_tower.loss_history            # mean BPR loss per epoch (a list)
n_epochs_recorded = len(losses)            # 60, the pinned default
first_loss = float(losses[0])
last_loss = float(losses[-1])
loss_decreased = bool(last_loss < first_loss)

# A reader WITH a learned embedding gets K recs (the score does not depend on the seen set; seen
# only EXCLUDES already-read books). A cold/unknown reader -- not in this list -- would get [].
warm_reader = two_tower.artifact()["reader_ids"][0]
warm_recs = [c.item_id for c in two_tower.retrieve(warm_reader, {"seen": seen_by[warm_reader]}, K)]
got_k_recs = bool(len(warm_recs) == K)

falling_loss_explanation = (
    "A falling BPR loss means the towers are learning to rank: epoch by epoch the positive books "
    "are pushed above their sampled negatives, so the mean -log sigmoid(s_pos - s_neg) shrinks. But "
    "a falling TRAINING loss alone does not prove the model generalizes -- it only says the model "
    "fits the pairs it was shown. Exercise 6 makes this concrete: with weight_decay=0 the train loss "
    "collapses toward 0 while validation hit@10 drops below CF, the signature of overfitting."
)

print(f"epochs recorded: {n_epochs_recorded}")
print(f"BPR loss {first_loss:.4f} -> {last_loss:.4f}  (decreased? {loss_decreased})")
print(f"warm reader {warm_reader} got {len(warm_recs)} recs")

assert n_epochs_recorded == 60, "the pinned default trains 60 epochs"
assert loss_decreased, "the BPR loss must fall from the first epoch to the last"
assert got_k_recs, "a reader with a learned embedding gets K recommendations"

## Exercise 4 - Register it and read the validation scoreboard (honestly)

Wrap the trained path as a first-class recommender and score it beside the rest of the book. Register the fitted `two_tower` in a `PathRegistry` (it must own the name `two-tower-v1`), then measure hit@10 on `val` for the two-tower, Unit 5's matrix factorization (MF), Unit 4's item-item CF, Unit 3's lexical path, and the random floor.

Read the numbers **honestly, and the honest reading is a real win this time**. The two-tower should land around hit@10 ~0.34 -- the **best path in the book so far**, above MF (~0.276), item-item CF (~0.252), lexical (~0.158) and the floor (~0.012). State that ordering plainly. Then be precise about *why*, so nobody mislearns "neural beats everything": the two-tower is the **same** reader-times-book dot product as MF. What changed is the training -- a pairwise **BPR ranking** objective with Adam mini-batches and **weight decay**, versus MF's pointwise logistic full-batch gradient descent -- and that ranking objective, *once regularized*, wins on top-k. In three or four sentences give the ordering and this honest explanation, and note that the win depends on weight decay (Exercise 6 shows `weight_decay=0` drops the two-tower below CF).

In [ ]:
registry = PathRegistry()
registry.register(two_tower)
artifact_name = two_tower.artifact_name()                 # 'two-tower-v1'
registered = bool(artifact_name == "two-tower-v1" and "two-tower" in registry)

# Fit the comparison paths. MF is the only other slow fit (~15 s); CF / lexical / random are fast.
# The two-tower itself is REUSED from Exercise 3 (no refit) -- one slow fit in this cell.
mf = MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)   # matrix factorization
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)       # collaborative filtering
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)  # surface-token matching
floor = RandomRetrievalPath(catalog_ids, seed=SEED)                      # the random floor


def val_hit(path):
    return run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    ).hit_rate_at_k


tt_hit = val_hit(two_tower)   # ~0.340 (the new best)
mf_hit = val_hit(mf)          # ~0.276
cf_hit = val_hit(item_item)   # ~0.252
lex_hit = val_hit(lexical)    # ~0.158
floor_hit = val_hit(floor)    # ~0.012

two_tower_is_best = bool(tt_hit > mf_hit > cf_hit > lex_hit > floor_hit)
beats_mf = bool(tt_hit > mf_hit)   # a real win, not "on par"

honest_reading = (
    "The two-tower (~0.340) is the BEST path in the book so far: tt > MF (~0.276) > item-item CF "
    "(~0.252) > lexical (~0.158) > the random floor (~0.012). But it is NOT that neural beats "
    "everything -- the two-tower is the SAME reader-times-book dot product as MF. What changed is "
    "the training: a pairwise BPR ranking objective with Adam mini-batches and weight decay, versus "
    "MF's pointwise logistic full-batch gradient descent, and that ranking objective wins on top-k "
    "once regularized. The win depends on weight decay -- Exercise 6 shows weight_decay=0 drops the "
    "two-tower below CF."
)

print(f"registered as {artifact_name!r}? {registered}")
print(f"two-tower hit@{K} = {tt_hit:.3f}  (new best)")
print(f"mf        hit@{K} = {mf_hit:.3f}")
print(f"item-item hit@{K} = {cf_hit:.3f}")
print(f"lexical   hit@{K} = {lex_hit:.3f}")
print(f"floor     hit@{K} = {floor_hit:.3f}")

assert registered, "the path must own and register as 'two-tower-v1'"
assert two_tower_is_best, "honest ordering: tt > MF > CF > lexical > floor"
assert beats_mf, "the two-tower really beats MF here (a win, not 'on par')"
assert tt_hit > 1.2 * cf_hit, "the two-tower clears item-item CF with room"

## Exercise 5 - Determinism: two seeded fits give the same answer

A trained model you cannot reproduce is not science. `TwoTowerRetrievalPath` makes every fit reproducible: it seeds `torch.manual_seed` (the embedding init) and a numpy RNG (the negative sampling and shuffling), turns on `torch.use_deterministic_algorithms(True)`, and pins `torch.set_num_threads(1)`. Two fits at the same seed must therefore agree. (The last two switches are process-global, so `fit` saves and restores their previous values -- it will not silently leave your later cells single-threaded.)

Fit a **second** two-tower at the same seed and compare it to the `two_tower` from Exercise 3. Confirm the learned embeddings are `allclose` and that the top-K ranking is **identical** for several readers -- that tolerance-based ranking match is the reproducibility gate (bit-identical weights happen to hold on CPU here too, a nice bonus, but ranking + `allclose` is the standard we hold the model to). In two or three sentences, explain why we gate on identical *ranking* plus `allclose` rather than demanding exactly equal floats.

In [ ]:
# A SECOND fit at the SAME seed (~15 s). The first fit is `two_tower`, reused from Exercise 3.
two_tower_again = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)

# The learned embedding matrices agree to floating-point tolerance.
readers_allclose = bool(np.allclose(two_tower.reader_embeddings, two_tower_again.reader_embeddings))
items_allclose = bool(np.allclose(two_tower.item_embeddings, two_tower_again.item_embeddings))

# The top-K ranking is IDENTICAL for the first few readers (the reproducibility gate). Empty seen
# is fine -- a warm reader is scored regardless of seen, which only excludes already-read books.
reader_ids = two_tower.artifact()["reader_ids"][:5]
rankings_identical = bool(all(
    [c.item_id for c in two_tower.retrieve(r, {"seen": set()}, K)]
    == [c.item_id for c in two_tower_again.retrieve(r, {"seen": set()}, K)]
    for r in reader_ids
))

# Bonus (holds on CPU here, but NOT the gate): the weights are even bit-identical.
bit_identical = bool(np.array_equal(two_tower.item_embeddings, two_tower_again.item_embeddings))

determinism_explanation = (
    "We gate on identical top-K RANKING plus allclose embeddings -- a tolerance-based standard -- "
    "rather than demanding exactly equal floats, because the thing a recommender actually serves is "
    "the ORDER of the results, and tiny float differences that never change that order are harmless. "
    "Insisting on bit-identical weights would be brittle (a different CPU, BLAS build or thread count "
    "can perturb the last bits) while buying nothing the ranking gate does not already guarantee. "
    "Bit-identity happens to hold here on a single-threaded CPU, so we keep it as a bonus check only."
)

print(f"reader embeddings allclose? {readers_allclose}")
print(f"item embeddings allclose?   {items_allclose}")
print(f"top-{K} rankings identical for 5 readers? {rankings_identical}")
print(f"bonus: weights bit-identical? {bit_identical}")

assert readers_allclose, "two seeded fits give allclose reader embeddings"
assert items_allclose, "two seeded fits give allclose item embeddings"
assert rankings_identical, "the reproducibility gate: identical top-K ranking across fits"

## Exercise 6 - Challenge: weight decay decides whether the two-tower wins

Here is the pivotal result of the whole unit. `weight_decay` is Adam's L2 penalty -- it gently shrinks the embeddings each step, discouraging the model from memorizing the exact train pairs. On this data it is the difference between the book's best path and a flop. Sweep `weight_decay` over `{0, 1e-5, 1e-4, 1e-3}`, holding everything else fixed, and measure val hit@10 for each. You should find a clear peak: `weight_decay=0` **overfits** (its BPR training loss collapses toward 0 while val hit@10 falls to ~0.18, **below** item-item CF ~0.252), `1e-4` peaks (~0.35 at these 20 epochs; the pinned 60-epoch default lands ~0.34), and `1e-3` **over-shrinks and collapses to ~0.17** -- as bad as no decay, but from the opposite failure (underfitting: its training loss barely falls). The curve has a sharp peak and fails on BOTH sides.

This cell does **four** fits, so it trains each for only `20` epochs instead of the pinned 60 (~5 s each, ~20 s total -- well under the 120 s/cell budget). Fewer epochs means a bit LESS overfitting, so `weight_decay=0` here (~0.18) sits slightly above the lesson's 60-epoch demo (~0.14) -- still below CF; the **ordering and the lesson are what matter**. In three or four sentences, explain the overfitting story (why a pairwise loss that memorizes train pairs generalizes *worse*), and note the bridge to Unit 9: feature towers and harder negatives are further ways to fight exactly this overfitting.

In [ ]:
# Four SMALL fits (20 epochs each instead of the pinned 60) so the whole sweep stays well under the
# 120 s/cell cap (~5 s per fit, ~20 s total). We trade a little accuracy for speed; the ORDERING
# across weight_decay values is the lesson, not the absolute numbers.
SWEEP_EPOCHS = 20
decays = [0.0, 1e-5, 1e-4, 1e-3]


def val_hit(path):   # same helper as Exercise 4
    return run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    ).hit_rate_at_k


sweep = {}   # weight_decay -> val hit@10
for wd in decays:
    path = TwoTowerRetrievalPath(weight_decay=wd, n_epochs=SWEEP_EPOCHS, seed=SEED).fit(
        rows, catalog=catalog_ids
    )
    sweep[wd] = val_hit(path)

hit_no_decay = sweep[0.0]      # overfits, BELOW item-item CF (~0.252); 1e-3 under-shrinks even lower
hit_best_decay = sweep[1e-4]   # the peak (the shipped default's weight decay)
decay_helps = bool(hit_best_decay > hit_no_decay)
no_decay_falls_below_cf = bool(hit_no_decay < 0.252)   # below item-item CF -- the overfitting signature

regularization_explanation = (
    "With weight_decay=0 the Adam optimizer is free to grow the embeddings until the BPR training "
    "loss collapses toward 0 -- it MEMORIZES the exact train pairs -- and that memorization "
    "generalizes WORSE: val hit@10 falls below item-item CF. weight_decay is Adam's L2 penalty: it "
    "gently shrinks the embeddings each step, so 1e-4 keeps the model general and lands at the "
    "book's best hit@10, while 1e-3 over-shrinks so hard the model underfits (loss barely falls) and collapses to ~0.17 -- as low as no decay, from the opposite direction. The bridge to Unit 9: feature towers "
    "(sharing information across items) and harder negatives are further ways to fight exactly this "
    "overfitting, so a learned retriever generalizes instead of memorizing."
)

for wd in decays:
    print(f"weight_decay={wd:<7}  val hit@{K} = {sweep[wd]:.3f}")
print(f"decay helps? {decay_helps}   weight_decay=0 below CF? {no_decay_falls_below_cf}")

assert decay_helps, "weight_decay=1e-4 must beat weight_decay=0"
assert no_decay_falls_below_cf, "weight_decay=0 overfits BELOW item-item CF (~0.252)"
assert hit_best_decay >= sweep[1e-3], "1e-4 is at or above the over-shrunk 1e-3 setting"

## Exercise 7 - Challenge: two-tower and MF are the same dot-product retriever

Unit 5's MF and this unit's two-tower are the **same retriever** wearing different clothes: each gives a reader and a book a vector in one shared space and scores them by a **dot product**. The only thing that differs is how the vectors are learned -- MF by pointwise logistic full-batch numpy gradient descent, the two-tower by pairwise BPR + Adam mini-batches + weight decay in PyTorch. Because they learn *different* vectors from the same log, they rank somewhat differently, which is exactly why the two-tower can top MF.

Make the equivalence concrete. Pull the two-tower's **learned** reader and item embedding matrices (torch-free, straight off the fitted path), rebuild one reader's top book **by hand** as the dot product `reader_emb . item_emb^T`, and confirm it matches what `two_tower.retrieve` returns -- the path's score *is* that dot product. Then measure the top-K **agreement** (overlap fraction) between the two-tower and MF for that reader: it should be partial (shared dot-product form, different learned vectors). In three or four sentences, describe the bridge -- pretrained (GloVe, Unit 7), learned (MF, Unit 5) and learned-neural (two-tower, Unit 8) are all dot-product retrievers over embeddings; only the *source* of the vectors differs.

In [ ]:
# The two-tower's LEARNED embeddings, as numpy (no torch needed to read or score them).
art = two_tower.artifact()
reader_mat = np.array(art["reader_embeddings"])   # (n_readers, 32)
item_mat = np.array(art["item_embeddings"])       # (n_books, 32)
tt_reader_ids = list(art["reader_ids"])
tt_item_ids = list(art["item_ids"])
reader_of = {r: i for i, r in enumerate(tt_reader_ids)}

example_reader = tt_reader_ids[0]
example_seen = seen_by[example_reader]

# By hand: score every catalog book by the learned dot product reader_emb . item_emb.
hand_scores = reader_mat[reader_of[example_reader]] @ item_mat.T   # one score per book column
hand_order = np.argsort(-hand_scores)                             # book columns best-first
hand_top_id = next(tt_item_ids[j] for j in hand_order if tt_item_ids[j] not in example_seen)

# What the path returns for the same reader -- also a dot product, over the SAME numpy weights.
path_top_id = rank(
    two_tower.retrieve(example_reader, {"seen": example_seen}, K), n=K, exclude=example_seen
)[0].item_id
hand_matches_path = bool(hand_top_id == path_top_id)

# Top-K agreement with MF (same dot-product FORM, different LEARNED vectors + objective).
tt_top = [c.item_id for c in rank(
    two_tower.retrieve(example_reader, {"seen": example_seen}, K), n=K, exclude=example_seen
)]
mf_top = [c.item_id for c in rank(
    mf.retrieve(example_reader, {"seen": example_seen}, K), n=K, exclude=example_seen
)]
agreement = len(set(tt_top) & set(mf_top)) / K
partial_agreement = bool(0.0 <= agreement < 1.0)

bridge_explanation = (
    "The by-hand dot product reproduces the path's top book exactly, because retrieval IS "
    "reader_emb . item_emb^T over the stored numpy weights -- no torch needed to score. The "
    "two-tower and MF share this dot-product FORM but learn DIFFERENT vectors (BPR + Adam + weight "
    "decay vs pointwise logistic full-batch GD), so their top-10 lists only partly overlap -- which "
    "is exactly how the two-tower can out-rank MF. GloVe (pretrained, Unit 7), MF (learned from the "
    "log, Unit 5) and the two-tower (learned by a neural network, Unit 8) are therefore all the same "
    "dot-product retriever over reader/book embeddings; only the SOURCE of the vectors differs."
)

print(f"hand top book {hand_top_id} == path top {path_top_id}? {hand_matches_path}")
print(f"two-tower vs MF top-{K} agreement for reader {example_reader}: {agreement:.2f}")

assert hand_matches_path, "the by-hand learned dot product reproduces the path's top book"
assert partial_agreement, "two-tower and MF share the dot-product form but learn different vectors"
assert reader_mat.shape[1] == item_mat.shape[1], "reader and item embeddings share one dot-product space"